# latent-riemannian-world — Quick Start

This notebook demonstrates the core features of  using a simple linear decoder.

## 1. Setup

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Install if needed
# pip install latent-riemannian-world

In [ ]:
from lrw.metric import PullbackMetric, BayesianMetric
from lrw.geodesic import GeodesicSolver, slerp_path
from lrw.transport import SchildsLadder, PoleLadder
from lrw.bayes import SVGD, RiemannianSGLD
from lrw.world import LatentStateSpace
from lrw.utils import riemannian_norm, manifold_assert_shape

print("All imports OK")

## 2. Define a Simple Decoder

In [ ]:
torch.manual_seed(42)

LATENT_DIM = 4
OUTPUT_DIM = 16

# Simple linear decoder for demonstration
W = torch.randn(OUTPUT_DIM, LATENT_DIM)

def decoder(z: torch.Tensor) -> torch.Tensor:
    """Linear decoder: z (B, D) -> x (B, M)"""
    return z @ W.T

print(f"Decoder: R^{LATENT_DIM} -> R^{OUTPUT_DIM}")

## 3. Pullback Metric

In [ ]:
metric = PullbackMetric(decoder=decoder)

z = torch.randn(4, LATENT_DIM)
G = metric.metric_tensor(z)  # (4, 4, 4)

print(f"Metric tensor shape: {G.shape}")
print(f"Symmetric: {torch.allclose(G, G.mT, atol=1e-5)}")
print(f"Positive definite: {(torch.linalg.eigvalsh(G) > 0).all()}")
print(f"Volume elements: {metric.local_volume_element(z)}")

## 4. Geodesic Interpolation vs SLERP

In [ ]:
solver = GeodesicSolver(metric=metric, n_steps=20, step_size=0.05)

z0 = torch.randn(1, LATENT_DIM)
z1 = torch.randn(1, LATENT_DIM)

# Geodesic path
geo_path = solver.interpolate(z0, z1, n_points=10)  # (10, 1, 4)

# SLERP path (baseline)
slerp = slerp_path(z0, z1, n_points=10)  # (10, 1, 4)

# Geodesic distance
dist = solver.geodesic_distance(z0, z1)
print(f"Geodesic distance: {dist.item():.4f}")
print(f"Geodesic path shape: {geo_path.shape}")
print(f"SLERP path shape: {slerp.shape}")

## 5. Parallel Transport

In [ ]:
v = torch.randn(1, LATENT_DIM) * 0.1  # style vector

# Schild's Ladder
schild = SchildsLadder(metric=metric, n_rungs=5)
v_schild = schild.transport(z0, z1, v)

# Pole Ladder (more accurate)
pole = PoleLadder(metric=metric, n_rungs=5)
v_pole = pole.transport(z0, z1, v)

G0 = metric.metric_tensor(z0)
G1 = metric.metric_tensor(z1)

norm_before = riemannian_norm(G0, v)
norm_schild = riemannian_norm(G1, v_schild)
norm_pole = riemannian_norm(G1, v_pole)

print(f"Norm before transport:  {norm_before.item():.4f}")
print(f"Norm after Schild:      {norm_schild.item():.4f}")
print(f"Norm after Pole Ladder: {norm_pole.item():.4f}")

## 6. Bayesian Metric

In [ ]:
# Ensemble of decoders (MC-Dropout simulation)
decoders = []
for i in range(8):
    torch.manual_seed(i)
    W_i = torch.randn(OUTPUT_DIM, LATENT_DIM)
    def make_decoder(W):
        def dec(z): return z @ W.T
        return dec
    decoders.append(make_decoder(W_i))

bayes_metric = BayesianMetric(decoder_ensemble=decoders)
G_bayes = bayes_metric.metric_tensor(z)
G_var = bayes_metric.metric_variance(z)

print(f"Bayesian metric shape:     {G_bayes.shape}")
print(f"Metric uncertainty shape:  {G_var.shape}")
print(f"Mean uncertainty:          {G_var.mean().item():.4f}")

## 7. World Model — Latent Trajectory

In [ ]:
state_space = LatentStateSpace(metric=metric, dt=0.05, noise_scale=0.01)

z0_traj = torch.randn(1, LATENT_DIM)
v0_traj = torch.randn(1, LATENT_DIM) * 0.1

states, velocities = state_space.rollout(z0_traj, v0_traj, n_steps=20)

print(f"Trajectory shape: {states.shape}")  # (21, 1, 4)
print(f"Start: {states[0].squeeze().tolist()}")
print(f"End:   {states[-1].squeeze().tolist()}")

## 8. Manifold Assertions

In [ ]:
from lrw.utils import manifold_assert_shape, manifold_assert_metric, manifold_assert_tangent

# Valid inputs
manifold_assert_shape(z, expected_dim=LATENT_DIM)
manifold_assert_metric(G)
manifold_assert_tangent(z, torch.randn_like(z))
print("All assertions passed")

# Invalid input example
try:
    manifold_assert_shape(torch.randn(4), expected_dim=LATENT_DIM)
except ValueError as e:
    print(f"Caught error: {e}")